***First Importing the important LIbraries for our project***
We will use kagglehub to directly download our dataset as it is quite large

In [1]:
!pip install kagglehub torch torchvision numpy matplotlib

In [2]:
import kagglehub
import os
import torch
import torchvision
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

print("Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(f"Dataset downloaded to: {dataset_path}")

Dataset downloaded to: C:\Users\samsa\.cache\kagglehub\datasets\xhlulu\140k-real-and-fake-faces\versions\2


***Exploring and preprocessing the data***
we can first have a look into our directory file because ideally we want to make sure our data is correct, we want to see how the fake vs real images are organised and separated in result will confirm the different classes 'real' or 'fake' the model will learn.

In [3]:
# Look into the data directory to see how the images and training sets are separated
data_dir = 'C:/Users/samsa/.cache/kagglehub/datasets/xhlulu/140k-real-and-fake-faces/versions/2/real_vs_fake/real-vs-fake'
print(f"Root data directory contains: {os.listdir(data_dir)}")
classes = os.listdir(data_dir + "/train")
print(f"Training classes found: {classes}")

Root data directory contains: ['test', 'train', 'valid']
Training classes found: ['fake', 'real']


***Now we need to define some image transformations because neural networks require the input images to be the same size, so we can resize all images to 64x64 using tt.Resize(64,64) and also use ToTesnor() to convert the data to a data format that the model is able to understand and learn better from.
It scales all the pixel values also from 0-255 to instead a value between 0.0-1.0.***

In [4]:
#Antialias= true is used to smooth out sharp and pixelated edges on diagonal lines or curves when images are being resized. 
from torchvision.datasets import ImageFolder
import torchvision.transforms as tt

train_tfms = tt.Compose([tt.Resize((64,64), tt.ToTensor(), antialias=True)])
valid_tfms = tt.Compose([tt.Resize((64,64), tt.ToTensor(), antialias=True)])

***We will now try and use the ImageFolder function from pytorch which is used to basically it automatically finds the images, assigns the correct labels etc on the subfolder names. For example it wil llabel fake for the fake images subfolder.***


In [5]:
# PyTorch datasets
train_ds = ImageFolder(data_dir+'/train', train_tfms)
valid_ds = ImageFolder(data_dir+'/test', valid_tfms)

print(f"No. of training examples: {len(train_ds)}")
print(f"No. of validation examples: {len(valid_ds)}")

No. of training examples: 100000
No. of validation examples: 20000


***Data Loading and Visualisation, we will here create our batch sizes and Data Loaders. Basically what Data Loader does here is that instead of having to fit and feed a large amount of data into the model. The DataLoader groups the data into smaller batches and feeds them batch by batch which is more memory efficient and better in general for learning for the model. This is also known as Stochastic Gradient Descent.***

In [6]:

from torch.utils.data import DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size=32
# Set num_workers=0 (or more) to speed up loading and pin_memory=True for faster GPU transfer
train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
valid_dl = DataLoader(valid_ds, batch_size*2, num_workers=0, pin_memory=(device.type == "cuda"))

for images, labels in train_dl:
    print(f"Images batch shape: {images.shape}")
    print(f"Labels batch shape: {labels.shape}")
    break

Images batch shape: torch.Size([32, 3, 64, 64])
Labels batch shape: torch.Size([32])


***We can now for our sanity create a function to check the batch of images from one DataLoader so that we can confirm that our image is being loaded, the transformations are correctly applied etc.***

In [ ]:
from torchvision.utils import make_grid

def show_batch(dl):
    for images, labels in dl:
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.set_xticks([]); ax.set_yticks([])
        ax.imshow(make_grid(images[:16], nrow=8).permute(1, 2, 0).clamp(0,1))
        break

show_batch(train_dl)